# Workflow prompt causal GPT models
Script for GPT4o-mini & GPT4 with a more complex prompt where two models discuss to make a prediction using langchain and openAI on Cloud service (Azure)

### Workflow-based classification approach

This notebook implements a two-stage classification workflow that combines two classifiers with different objectives:

1. **Classifier 1** is optimized for high recall while maintaining a high F1 score. Its purpose is to minimize the risk of missing potentially relevant cases.
2. **Classifier 2** favors precision over recall while also maintaining a high F1 score. It is used to reassess cases identified as positive by the first classifier.

The classifiers are combined as follows:

* If **Classifier 1 predicts class 0**, the prediction is accepted as the final classification.
* If **Classifier 1 predicts class 1**, the case is passed to Classifier 2:

  * If both classifiers predict **class 1**, class 1 is accepted as the final classification.
  * If the classifiers **disagree**, both classifiers are asked to provide reasoning for their respective decisions. Classifier 1 then reassesses the case using these explanations and makes the final classification.

This workflow prioritizes sensitivity in the initial screening while using a second, more precision-oriented classifier and explicit reasoning to further assess potentially relevant cases.


In [0]:

import re
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from langchain_openai import AzureChatOpenAI



## Loading MAUDE Data and preprocessing

In [0]:
# Data cleaning function - combined
def standardization(sent: str) -> str:
    '''
    Input: raw reviews (string)
    Output: cleaned & standardized reviews (string)
    '''
    # Convert to lowercase, remove unwanted patterns, and remove non-alphanumeric characters
    sent = re.sub(r'[^0-9a-zA-Z-ZäöüÄÖÜßéóƒÚâèåèñéçýáúåí\s]', '', sent.lower())  
    # Remove specific MAUDE patterns
    sent = re.sub(r'\(b\)\(6\)|\(b\) \(6\)|\(b\)\(4\)|\(b\) \(4\)|\[rs\]\.\\n', '', sent)
    # Replace multiple spaces and strip leading/trailing whitespaces
    sent = re.sub(r'\s+', ' ', sent).strip()  
    
    return sent

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Input: DataFrame with column 'text' 
    Output: Cleaned DataFrame with standardized text
    '''
    # Standardize text by applying the standardization function to each row
    df["text"] = df["text"].apply(standardization) 

    return df

# params
max_words            = 2000

## Load and preprocess dataset
filename = "data/cybersecurity_annotated_data.pq"
df = pd.read_parquet(filename)
df = clean_dataframe(df[['ID', 'text', 'label']].dropna())
df['text'] = df['text'].apply(lambda x : ' '.join(x.split(' ')[:max_words]))
df.head()


## Split data

Splitting the data in 5 chunks as used in the "traditional" models for foldwise comparability

In [0]:

# Run the prompt on the same sets as the traditional models to allow for comparability
X = df["text"]
y = df["label"]

# Set up 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []
FoldsDict = {}
for n, (train_index, test_index) in enumerate(kf.split(X)):
    
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    thisFold = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }
    FoldsDict[n] = thisFold 
 



In [0]:
Fold = 4
x = FoldsDict[Fold]["X_test"]
y = FoldsDict[Fold]["y_test"] 

## Setup the openAI API

In [0]:
# Initialize Header
open_api_headers = {
    "client_id": "yourID",
    "client_secret": "yourSecret"
}
                      
open_api_base = "YOURAPI"


# initialize the Langchain Adapter for Azure Open AI
model1 = AzureChatOpenAI(
    model="gpt-4o",
    default_headers=open_api_headers,
    openai_api_type="azure",
    azure_endpoint=open_api_base,
    openai_api_key= "not_relevant",
    openai_api_version="2024-06-01"
)

model2 = AzureChatOpenAI(
    model="gpt-4o-mini",
    default_headers=open_api_headers,
    openai_api_type="azure",
    azure_endpoint=open_api_base,
    openai_api_key= "not_relevant",
    openai_api_version="2024-06-01"
)

## Used Prompts


In [0]:
#use this in step 1
PromptVersion = "Prompt1C1C0L"
sys_prompt_firststep = """Role:
You are a helpful assistant tasked with classifying text based on its relevance to cybersecurity.

Instructions:
A text is relevant to cybersecurity if it contains any of the following topics:
    - Hacking, phishing, unauthorized access, data breaches, malware, or ransomware.
    - Unauthorized access to systems, devices, networks, or data.
    - Loss or theft of sensitive information.
    - Lack of cybersecurity protection such as firewalls and antivirus software.
    - Suspected cybersecurity attacks, even if unverified.
    - Fraudulent behavior related to networks, the internet, or computers.
    - Switched off firewalls.
    - Technically implausible scenarios.
A text is not cybersecurity-relevant if it:
    - Mentions software updates, system backup modes, general system errors or malfunctions.
    - Describes primarily a medical issue, surgical procedure, or medical case report.
    - Mentions "hacking" in the context of coughing or mechanical issues.
    - Does not mention cybersecurity.
    - Relates to a defective device unrelated to cybersecurity.
    - Describes fraudulent behavior by individuals or companies.
    - Lists many possible root causes.
    - Is not classifiable by the given rules.

Answer format:
Respond ONLY with either '0' or '1'.
Respond with '0' if the report is not cybersecurity relevant.
Respond with '1' if the report is cybersecurity relevant.

"""

In [0]:
PromptVersion = "Prompt2C1C0L"
sys_prompt_2nd_step = """
Your role is Cybersecurity incident classification expert.
You classify whether the report describes a cybersecurity incident case based on the following instructions:

    Classify as '1' if the report mentions actions like hacking, phishing, unauthorized access, data breaches, malware, or ransomware.
    Classify as '1' if the report states unauthorized access to systems, devices, networks, or data.
    Classify as '1' if the report contains info about loss or theft of sensitive information.
    Classify as '1' if the report mentions lacking cybersecurity protection such as firewalls and antivirus software.
    Classify as '1' if the patient suspects cybersecurity attacks, even if unverified.
    Classify as '1' if the report describes fraudulent behaviour related to networks, internet or computers.
    Classify as '1' if the report mentions switched off firewalls.
    Classify as '1' if the report content appears technically not plausible.

    Classify as '0' if the report contains software updates, system backup modes, or general malfunctions.
    Classify as '0' if the report describes primarily a medical issue or surgical procedure.
    Classify as '0' if the report contains information about a general system error or malfunction.
    Classify as '0' if the report does not mention cybersecurity-related concerns.
    Classify as '0' if the report relates to a defective device unrelated to cybersecurity.
    Classify as '0' if the report describes fraudulent behaviour by businesses or by companies.
    Classify as '0' if the report lists many possible root causes.

    Do not base the classification on the word "hacking" if it relates to coughing or when describing a mechanical issue.
    Respond ONLY with '0' or '1'."""

## Classification

In [0]:
def cleanclassif(classification):
    if len(str(classification)) == 3: #string cleaning of answer 
        classification = str(classification)[1]
    elif len(str(classification)) > 3:
        classification = 4
    try : 
        classification = int(classification)
    except ValueError:
        classification = 2
    return classification

In [0]:

import time
#PromptVersion = "Prompt1C1C0L"
# gtp4 fewshot with "Prompt1C1C0L" as recall positive classifier
# gpt4mini prompt2c1c0L as precison classifier  sys_prompt_firststep

RES_DF = pd.DataFrame()

prompt_template = ChatPromptTemplate([
    ("system", """ {a_sys_prompt} """),
    ("user", """ {a_human_prompt} """  + """ {report} """ ) ])
                                     
# Create the LLMChain for classification
classifier_chain = LLMChain(llm=model1, prompt=prompt_template) #gpt4, 
classifier_chain2 = LLMChain(llm=model2, prompt=prompt_template)#gpt4o_mini
a_human_prompt = "Please classify if following report *describes* cybersecurity case or *does not describe* a cybersecurity case. Here is the report:"
human_prompt_pos = "Please justify why the following report *describes* a cybersecurity incident case. Answer with 'none' if no such reason can be found. Here is the report:"
human_prompt_neg =  "Please justify why the following report *does not describe* a cybersecurity incident case. Answer with 'none' if no such reason can be found. Here is the report:"

for n, text_to_classify in enumerate(x):

    time.sleep(5)
    print(f"+++++++++++++++++ this is number {n}" )
    try:
        classification = classifier_chain.run({"a_sys_prompt": sys_prompt_firststep, "a_human_prompt": a_human_prompt, "report": text_to_classify })
    except:
        classification = 3 #if a content filter kicks in, eg violence
        print(f"Maybe this text goes against some rules: {text_to_classify}")

    #print(f"Classification before Corrections: {classification}")

    classification = cleanclassif(classification) #str
    print(f"label: {y.iloc[n]}")
    print(f"Classification1: {classification}")
    if classification == 0:
        pass
    if classification == 1: #now ask in more detail   
        try:
            classification2 = classifier_chain2.run({"a_sys_prompt": sys_prompt_2nd_step, "a_human_prompt": a_human_prompt,"report": text_to_classify })
        except:
            classification2 = 4 #if a content filter kicks in, eg violence
            print(f"Maybe this text goes against some rules: {text_to_classify}")
        
        classification2 = cleanclassif(classification2)
        if classification2 == classification:
            print("#The two classifiers agree")
            #classification = 1 #is anyhow
        if classification2 != classification:
            print("The classifiers disagree")  
            # reasons from the first classifier for making it a 1
            reason = classifier_chain.run({"a_sys_prompt": sys_prompt_firststep, "a_human_prompt": human_prompt_pos, "report": text_to_classify })
            print("Reason1:", reason)
            # reason for second classifier for making it a 0
            reason2 = classifier_chain2.run({"a_sys_prompt": sys_prompt_2nd_step, "a_human_prompt": human_prompt_neg, "report": text_to_classify })
            print("Reason2:", reason2)
            
            # discussiong it ... 
            Discuss_prompt = """You are given: 
            - Reason 1: Why the report *describes* a cybersecurity incident case.
            - Reason 2: Why the report *does not describe* a cybersecurity incident case..
            - The full report text.
             
            Instructions:
            Work step by step.
            1.Step: Evaluate both reasons in the context of the report.
            2.Step: Decide which reason is more strongly supported by the report.
            3.Step: Based on your evaluation, follow the rules in the system prompt and classify the report as either :
                - '1' → Cybersecurity relevant
                - '0' → Not cybersecurity relevant
                
            Input:
            Reason 1: """ + reason + """
            Reason 2: """ + reason2 + """
            
            Output:
            Answer with either '1' or '0'.
            
            Here is the report: """


            try:
                classification_f = classifier_chain.run({"a_sys_prompt": sys_prompt_firststep, "a_human_prompt": Discuss_prompt, "report": text_to_classify })
                print("result:", classification_f)
                classification = cleanclassif(classification_f)
                classification = int(classification)
            except:
                classification = 5
                print(f"Maybe this text goes against some rules: {text_to_classify}")

    
    RES_DF = RES_DF._append({"report": text_to_classify, "label":y.iloc[n], "predict": classification },ignore_index=True)

#save to excel

RES_DF.to_excel( PromptVersion + "Fold_" + str(Fold) + ".xlsx", index = False) #depends on the prompt cell which you ran

## Evaluate the results

In [0]:
def eval_model(y_test,y_pred):
    # value counts of the predicted labels
    print(RES_DF["predict"].value_counts())
  
    # Evaluate the model
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, digits= 3))

In [0]:
eval_model(RES_DF["label"].values ,RES_DF["predict"].values)